# Study of the network of Twitch gamers

OULD AKLOUCHE Sarah, PINATEL Olivier

## Imports

In [2]:
%pip install networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [42]:
import networkx as nx
import pandas as pd
# pip install networkx python-louvain pandas
import community as community_louvain
# pip install igraph leidenalg pandas
import igraph as ig
import leidenalg as la
from sklearn.metrics import normalized_mutual_info_score

## Reading the data

In [ ]:
# Lire les données
df=pd.read_csv("data/large_twitch_features.csv")
df2=pd.read_csv("data/large_twitch_edges.csv")

In [ ]:
# Création du graphe
G = nx.from_pandas_edgelist(df2, "numeric_id_1", "numeric_id_2", create_using=nx.Graph())

In [ ]:
# Features disponibles
df

,views,mature,life_time,created_at,updated_at,numeric_id,dead_account,language,affiliate
0,7879,1,969,2016-02-16,2018-10-12,0,0,EN,1
1,500,0,2699,2011-05-19,2018-10-08,1,0,EN,0
2,382502,1,3149,2010-02-27,2018-10-12,2,0,EN,1
3,386,0,1344,2015-01-26,2018-10-01,3,0,EN,0
4,2486,0,1784,2013-11-22,2018-10-11,4,0,EN,0
...,...,...,...,...,...,...,...,...,...
168109,4965,0,810,2016-07-20,2018-10-08,168109,0,EN,0
168110,4128,1,2080,2013-01-31,2018-10-12,168110,0,EN,0
168111,3545,0,1797,2013-11-08,2018-10-10,168111,0,EN,1
168112,892736,1,2135,2012-12-07,2018-10-12,168112,0,EN,0


# Label propagation

En premier, nous appliquons l'algorithme de Label propagation pour avoir une première idée de la structure possible du graphe.

In [ ]:
# Tourne en 5 minutes
communities = nx.algorithms.community.label_propagation_communities(G)

partition_lpa = {}
for i, comm in enumerate(communities):
    for node in comm:
        partition_lpa[node] = i

In [33]:
df_nodes_lpa = pd.DataFrame(list(partition_lpa.items()), columns=["node", "community_lpa"])

In [34]:
df_nodes_lpa["community_lpa"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78])

In [40]:
df_nodes_lpa["community_lpa"].value_counts()

community_lpa
0     160091
2       3143
1       2580
6        898
5        633
       ...  
74         2
75         2
76         2
77         2
78         2
Name: count, Length: 79, dtype: int64

79 communautés, avec une communautées très majoritaire.

In [ ]:
# On calcule la modularité afin de pouvoir comparer avec la méthode suivante
mod = community_louvain.modularity(partition_lpa, G)
print("Modularité Label Propagation:", mod)

Modularité Label Propagation: 0.03567630678997075


# Louvain

In [45]:
# Met 12 minutes à tourner
partition = community_louvain.best_partition(
    G,
    resolution=1.0,
    random_state=42
)

In [46]:
df_nodes_louvain = pd.DataFrame(list(partition.items()), columns=["node", "community"])

In [47]:
df_nodes_louvain["community"].value_counts()

community
3     47419
1     34678
0     24572
2     24146
11     8779
13     6219
5      5080
4      5016
8      4608
29     2010
9      1169
16      826
14      729
15      669
6       539
17      501
10      405
7       367
18      344
22        7
25        5
24        4
19        3
31        3
23        2
21        2
20        2
26        2
27        2
28        2
30        2
12        2
Name: count, dtype: int64

In [48]:
df_nodes_louvain["community"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 29, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 12])

31 communautés, dont 19 avec plus de 10 utilisateurs.

In [11]:
mod = community_louvain.modularity(partition, G)
print("Modularité:", mod)

Modularité: 0.4179872537787052


Bien meilleure modularité que label propagation, le graphe semble effectivement avoir une structure en communauté exploitable, mais celles ci sont détectables au niveau global et pas vraiment au niveau local.

# Leiden (mais n'utilise pas networkx donc à voir si on garde ou pas)

In [20]:
g = ig.Graph.TupleList(
    df2.itertuples(index=False),
    directed=False
)

In [ ]:
partition_leiden = la.find_partition(
    g,
    la.RBConfigurationVertexPartition,
    resolution_parameter=1.0
)

In [ ]:
membership = partition_leiden.membership

In [29]:
df_nodes_leiden = pd.DataFrame({
    "node": range(len(membership)),
    "community_leiden": membership
})

In [31]:
df_nodes_leiden["community_leiden"].unique()

array([ 1,  4,  0,  3,  2,  5, 10,  8, 16,  9, 12,  6, 11,  7, 14, 15, 13,
       17, 18])

In [32]:
df_nodes_leiden["community_leiden"].value_counts()

community_leiden
0     41765
1     30191
2     20603
3     16963
4     13609
5      9109
6      8785
7      6251
8      5094
9      4630
10     4146
11     2165
12     1167
13      841
14      727
15      671
16      545
17      503
18      349
Name: count, dtype: int64

19 communautés, toutes avec plus de 300 utilisateurs.

In [24]:
mod = g.modularity(membership)
print("Modularité Leiden:", mod)

Modularité Leiden: 0.4250376321226498


# Comparaison à l'aide de la Mutual Entropy (Normalized Mutual Information)
On utilise la NMI pour comparer ces trois algorithmes (a priori LPA est déjà éliminé), on les compare 2 à 2, donc on va obtenir une matrice de similarité. 
Ne marche pas pour le moment, à reprendre.

In [54]:
# Ordonner les noeuds
nodes = list(G.nodes())

#labels_lpa = [df_nodes_lpa[n] for n in nodes]
#labels_louvain = [df_nodes_louvain[n] for n in nodes] 
labels_leiden = list(membership)

In [56]:
assert len(df_nodes_lpa) == len(nodes)
assert len(df_nodes_louvain) == len(nodes)
assert len(labels_leiden) == len(nodes)

In [58]:
nmi_lpa_louvain = normalized_mutual_info_score(df_nodes_lpa, df_nodes_louvain)
#nmi_lpa_leiden = normalized_mutual_info_score(df_nodes_lpa, labels_leiden)
#nmi_louvain_leiden = normalized_mutual_info_score(df_nodes_louvain, labels_leiden)

print(f"NMI LPA vs Louvain : {nmi_lpa_louvain:.3f}")
#print(f"NMI LPA vs Leiden : {nmi_lpa_leiden:.3f}")
#print(f"NMI Louvain vs Leiden : {nmi_louvain_leiden:.3f}")

ValueError: labels_true must be 1D: shape is (168114, 2)